In [1]:
pip install pandas numpy statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 52.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 46.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 46.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [statsmodels] [statsmodels]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
os.chdir("..")

print(os.listdir())

combine = pd.read_csv("2025 Combine Results.csv")

['2025 Player Rankings.csv', '2025 Combine Results.csv', 'README.md', '.git', 'BSANFinal.ipynb']


In [5]:
combine["Pick"] = combine["Drafted (tm/rnd/yr)"].str.extract(
    r'(\d+)(?=th pick|st pick|nd pick|rd pick)'
).astype(float)

In [6]:
combine = combine.replace(-9999, np.nan)
combine = combine.replace(r"^\s*$", np.nan, regex=True)

In [7]:
model_df = combine[[
    "Pick",
    "40yd",
    "Wt",
    "Vertical",
    "Bench"
]].copy()

In [8]:
for col in ["Pick", "40yd", "Wt", "Vertical", "Bench"]:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

In [9]:
model_df = model_df.dropna()

print(model_df.shape)
print(model_df.head())

(32, 5)
     Pick  40yd   Wt  Vertical  Bench
2    65.0  4.95  305      31.5   28.0
5    61.0  4.43  195      32.5   13.0
11  176.0  4.62  258      35.5   19.0
14  156.0  4.63  232      38.5   21.0
27  193.0  4.52  214      35.0   16.0


In [10]:
X = model_df[["40yd", "Wt", "Vertical", "Bench"]]
y = model_df["Pick"]

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   Pick   R-squared:                       0.219
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     1.891
Date:                Thu, 23 Apr 2026   Prob (F-statistic):              0.141
Time:                        01:19:31   Log-Likelihood:                -165.16
No. Observations:                  32   AIC:                             340.3
Df Residuals:                      27   BIC:                             347.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -975.0904    571.966     -1.705      0.1

In [11]:
model_df["predicted_pick"] = model.predict(X)
print(model_df[["Pick", "predicted_pick"]].head(10))

     Pick  predicted_pick
2    65.0      125.433368
5    61.0      111.379755
11  176.0       93.812863
14  156.0      123.527804
27  193.0      115.031469
34  129.0      107.997423
68   35.0       67.702173
70  114.0      104.216907
75  137.0      173.084220
96   22.0       89.395411


In [12]:
model_df = combine[[
    "Pick",
    "40yd",
    "Wt",
    "Vertical",
    "Bench",
    "Pos"
]].copy()

In [13]:
model_df = pd.get_dummies(model_df, columns=["Pos"], drop_first=True)

In [14]:
for col in ["Pick", "40yd", "Wt", "Vertical", "Bench"]:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

In [15]:
model_df = model_df.dropna()

print(model_df.shape)

(32, 19)


In [16]:
X = model_df.drop(columns="Pick")
y = model_df["Pick"]

In [17]:
X = X.astype(float)
y = y.astype(float)

In [18]:
import statsmodels.api as sm

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

In [19]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   Pick   R-squared:                       0.502
Model:                            OLS   Adj. R-squared:                  0.227
Method:                 Least Squares   F-statistic:                     1.830
Date:                Thu, 23 Apr 2026   Prob (F-statistic):              0.116
Time:                        01:19:53   Log-Likelihood:                -157.97
No. Observations:                  32   AIC:                             339.9
Df Residuals:                      20   BIC:                             357.5
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1746.0785    656.989     -2.658      0.0

In [20]:
model_df["predicted_pick"] = model.fittedvalues.values

In [21]:
model_df[["Pick", "predicted_pick"]].head(10)

,Pick,predicted_pick
2,65.0,84.253955
5,61.0,89.882192
11,176.0,152.189404
14,156.0,135.307645
27,193.0,128.797702
34,129.0,108.888769
68,35.0,54.438295
70,114.0,110.203992
75,137.0,167.759763
96,22.0,86.726327


In [22]:
model_df["percentile"] = pd.qcut(model_df["Pick"], 10, labels=False)

In [23]:
def assign_group(p):
    if p == 0:
        return "Top 10%"
    elif p == 1:
        return "Next 10%"
    elif p == 2:
        return "Next 10% After That"
    else:
        return "Remaining 70%"

In [24]:
model_df["group"] = model_df["percentile"].apply(assign_group)

In [25]:
group_summary = model_df.groupby("group").apply(
    lambda x: pd.Series({
        "Count": len(x),
        "Average Actual Pick": x["Pick"].mean(),
        "Average Predicted Pick": x["predicted_pick"].mean(),
        "Mean Absolute Error": abs(x["Pick"] - x["predicted_pick"]).mean()
    })
).reset_index()

In [26]:
group_order = ["Top 10%", "Next 10%", "Next 10% After That", "Remaining 70%"]

group_summary["group"] = pd.Categorical(
    group_summary["group"],
    categories=group_order,
    ordered=True
)

group_summary = group_summary.sort_values("group")
print(group_summary)

                 group  Count  Average Actual Pick  Average Predicted Pick  \
3              Top 10%    4.0            41.500000               77.362961   
0             Next 10%    3.0            64.333333               95.021303   
1  Next 10% After That    3.0            74.666667               90.546745   
2        Remaining 70%   22.0           134.863636              121.992910   

   Mean Absolute Error  
3            35.862961  
0            30.687970  
1            15.880078  
2            28.763453  


In [29]:
rankings = pd.read_csv("2025 Player Rankings.csv", encoding="latin1")
rankings.columns = rankings.columns.str.strip()

In [30]:
print(rankings.head())
print(rankings["Name"].nunique())

   Season       Id             Name Position                 Team Conference  \
0    2025   550577     Jordan Brown       WR         Fresno State        MWC   
1    2025  3116099   Monte Harrison       WR             Arkansas        SEC   
2    2025  3149911   Jordan Jackson       WR  Arkansas-Pine Bluff        NaN   
3    2025  3949586      Jake Taylor       TE                 Duke        ACC   
4    2025  4241676  Kordell Rodgers       TE       Texas Southern        NaN   

   AveragePPA All  AveragePPA Pass  AveragePPA Rush  AveragePPA FirstDown  \
0           0.857            0.857              NaN                 0.829   
1           0.670            0.670              NaN                 0.670   
2           0.686            0.686              NaN                 0.717   
3           0.093            0.093              NaN                -0.837   
4           0.507            0.507              NaN                 0.650   

   ...  AveragePPA PassingDowns  TotalPPA All  TotalPPA 

In [31]:
def clean_name(name):
    name = str(name).lower()
    name = name.replace(".", "")
    name = name.replace(",", "")
    name = name.replace(" jr", "")
    name = name.replace(" sr", "")
    name = name.replace(" iii", "")
    name = name.replace(" ii", "")
    return name.strip()

combine["player_clean"] = combine["Player"].apply(clean_name)
rankings["Name_clean"] = rankings["Name"].apply(clean_name)

In [32]:
df = pd.merge(
    combine,
    rankings,
    left_on="player_clean",
    right_on="Name_clean",
    how="inner"
)

print(df.shape)

(3, 41)
